# MotionJSON Local UI with hosted and local SAM setup

This notebook launches the MotionJSON Local UI inside a Google Colab runtime and opens `/ui/` through Colab's built-in notebook port proxy. It is for connecting real SAM providers from the UI: local SAM2, local SAM3, Replicate SAM2 video, Roboflow SAM3, Fal SAM3 image, or custom SAM2/SAM3-compatible endpoints.

No public tunnel is started. Hosted provider keys are read from Colab userdata when available, with interactive fallback. You can also leave them blank and paste temporary credentials into the UI Model Connections form. Do not save private videos, provider credentials, or shared notebook outputs containing secrets.

Local SAM3 has two separate paths:

- `/content/sam3` is the cloned official SAM3 source/package directory. It lets Python import `sam3`, but it is not a model checkpoint.
- `SAM3_LOCAL_MODEL` must be a real local checkpoint file path, usually a Hugging Face cache path ending in `sam3.pt`, such as `/root/.cache/huggingface/hub/models--facebook--sam3/snapshots/<hash>/sam3.pt`.

If you prefer hosted SAM3, skip the local SAM3 package and checkpoint cells. Roboflow SAM3 and Fal SAM3 image are configured from Model Connections and do not require `SAM3_LOCAL_MODEL`.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

repo_url = "https://github.com/ptse8204/json-animated-video.git"
workdir = Path("/content/json-animated-video")

if not workdir.exists():
    subprocess.run(["git", "clone", repo_url, str(workdir)], check=True)
else:
    subprocess.run(["git", "-C", str(workdir), "fetch", "--depth", "1", "origin", "main"], check=True)
    subprocess.run(["git", "-C", str(workdir), "checkout", "main"], check=True)
    subprocess.run(["git", "-C", str(workdir), "pull", "--ff-only"], check=True)

os.chdir(workdir)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", ".[ui,hosted-segmentation,hosted-sam3,hosted-sam-vendors]"],
    check=True,
)


In [ ]:
print("Python:", sys.version.split()[0])
try:
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("CUDA device:", torch.cuda.get_device_name(0))
    else:
        print("For local SAM2/SAM3, switch Colab to a GPU runtime before installing model packages.")
except Exception as exc:
    print("PyTorch is not importable yet:", type(exc).__name__)
    print("Hosted providers can still be linked. Local SAM setup needs torch plus the official SAM package.")


In [ ]:
import re
from getpass import getpass

SAM3_SOURCE_DIR = Path("/content/sam3")
SAM3_HF_REPO_ID = "facebook/sam3"
SAM3_CHECKPOINT_FILENAME = "sam3.pt"
SAM3_HF_CACHE_DIR = Path.home() / ".cache" / "huggingface" / "hub" / "models--facebook--sam3"

def colab_user_secret(name: str) -> str:
    try:
        from google.colab import userdata
        value = userdata.get(name)
    except Exception:
        value = None
    return (value or "").strip()

def is_probably_hf_repo_id(value: str) -> bool:
    value = str(value or "").strip()
    return bool(re.fullmatch(r"[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+", value)) and not value.startswith(("/", ".", "~"))

def friendly_size(path: Path) -> str:
    try:
        size = path.stat().st_size
    except OSError:
        return "unknown size"
    units = ["bytes", "KB", "MB", "GB", "TB"]
    value = float(size)
    for unit in units:
        if value < 1024 or unit == units[-1]:
            return f"{value:.1f} {unit}" if unit != "bytes" else f"{int(value)} bytes"
        value /= 1024
    return f"{size} bytes"

def find_sam3_checkpoint_candidates(paths: list[Path]) -> list[Path]:
    candidates: list[Path] = []
    seen: set[str] = set()
    for root in paths:
        root = Path(root).expanduser()
        if root.is_file() and root.name == SAM3_CHECKPOINT_FILENAME:
            matches = [root]
        elif root.exists() and root.is_dir():
            matches = list(root.rglob(SAM3_CHECKPOINT_FILENAME))
        else:
            matches = []
        for candidate in matches:
            if candidate.is_file():
                key = str(candidate.resolve())
                if key not in seen:
                    seen.add(key)
                    candidates.append(candidate)
    return sorted(candidates, key=lambda p: p.stat().st_mtime if p.exists() else 0, reverse=True)

def set_and_validate_sam3_local_model(path: str | Path) -> Path:
    raw = str(path or "").strip()
    if not raw:
        raise ValueError("SAM3_LOCAL_MODEL is empty. Use a local sam3.pt checkpoint file path.")
    if is_probably_hf_repo_id(raw):
        raise ValueError(
            f"{raw!r} is a Hugging Face repo id, not a local file path. "
            "Download or resolve facebook/sam3 sam3.pt first, then use the returned local path."
        )
    model_path = Path(raw).expanduser()
    if str(model_path).rstrip("/") == str(SAM3_SOURCE_DIR):
        candidates = find_sam3_checkpoint_candidates([model_path])
        if candidates:
            raise ValueError(
                f"{model_path} is the cloned SAM3 source/package directory. "
                f"Use the checkpoint file instead: {candidates[0]}"
            )
        raise ValueError(
            f"{model_path} is the cloned SAM3 source/package directory, not the checkpoint. "
            "Run the checkpoint resolver cell or paste a real sam3.pt file path."
        )
    if model_path.is_dir():
        candidates = find_sam3_checkpoint_candidates([model_path])
        if candidates:
            print(f"{model_path} is a directory; using checkpoint file {candidates[0]}")
            model_path = candidates[0]
        else:
            raise ValueError(f"{model_path} is a directory and no sam3.pt checkpoint was found inside it.")
    if not model_path.exists():
        raise ValueError(f"SAM3_LOCAL_MODEL path does not exist: {model_path}")
    if model_path.name != SAM3_CHECKPOINT_FILENAME:
        print(f"Warning: expected a file named {SAM3_CHECKPOINT_FILENAME}; got {model_path.name}.")
    os.environ["SAM3_LOCAL_MODEL"] = str(model_path)
    return model_path

def print_sam3_path_help(model_path: Path | None = None) -> None:
    print("SAM3 local path guide:")
    print("- /content/sam3 is the official SAM3 source/package directory, not the checkpoint path.")
    print("- facebook/sam3 is the Hugging Face repo id, not a local model path.")
    print("- SAM3_LOCAL_MODEL must be a local checkpoint file path ending in sam3.pt.")
    if model_path:
        print("Copy these values into Model Connections -> SAM3 local:")
        print(f"  provider: SAM3 local")
        print(f"  model path: {model_path}")
        print(f"  device: {os.environ.get('SAM3_LOCAL_DEVICE', 'cuda')}")

for env_name in ["ROBOFLOW_API_KEY", "REPLICATE_API_TOKEN", "FAL_KEY", "HF_TOKEN"]:
    value = colab_user_secret(env_name)
    if not value:
        value = getpass(f"{env_name} (leave blank to skip): ").strip()
    if value:
        os.environ[env_name] = value

if os.environ.get("HF_TOKEN"):
    os.environ["HUGGINGFACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]

existing_sam3_path = colab_user_secret("SAM3_LOCAL_MODEL") or os.environ.get("SAM3_LOCAL_MODEL", "")
if existing_sam3_path:
    try:
        resolved_existing_path = set_and_validate_sam3_local_model(existing_sam3_path)
        print("SAM3_LOCAL_MODEL is already configured and exists.")
        print_sam3_path_help(resolved_existing_path)
    except ValueError as exc:
        os.environ.pop("SAM3_LOCAL_MODEL", None)
        print("Existing SAM3_LOCAL_MODEL is not usable:", exc)
        print_sam3_path_help()
else:
    print("No SAM3_LOCAL_MODEL path configured yet. Use the SAM3 checkpoint resolver cell after optional package setup.")

configured_secret_names = [name for name in ["ROBOFLOW_API_KEY", "REPLICATE_API_TOKEN", "FAL_KEY", "HF_TOKEN", "HUGGINGFACE_HUB_TOKEN"] if os.environ.get(name)]
configured_path_names = [name for name in ["SAM3_LOCAL_MODEL"] if os.environ.get(name)]
print("Configured secret names (values hidden):", configured_secret_names or "none")
print("Configured local path names:", configured_path_names or "none")


## Optional local SAM2 setup

Use local SAM2 for `Trace one object` when you want point/box-prompted video segmentation inside this Colab runtime. Run this only after selecting a GPU runtime. The official install and checkpoint commands are shown in the UI Model Connections panel too.

In [ ]:
RUN_LOCAL_SAM2_SETUP = False

if RUN_LOCAL_SAM2_SETUP:
    sam2_dir = Path("/content/sam2")
    if not sam2_dir.exists():
        subprocess.run(["git", "clone", "https://github.com/facebookresearch/sam2.git", str(sam2_dir)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(sam2_dir)], check=True)
    ckpt_dir = sam2_dir / "checkpoints"
    subprocess.run(["bash", "download_ckpts.sh"], cwd=str(ckpt_dir), check=True)
    os.environ.setdefault("SAM2_LOCAL_CHECKPOINT", str(ckpt_dir / "sam2.1_hiera_large.pt"))
    os.environ.setdefault("SAM2_LOCAL_CONFIG", "configs/sam2.1/sam2.1_hiera_l.yaml")
    os.environ.setdefault("SAM2_LOCAL_DEVICE", "cuda")
else:
    print("Set RUN_LOCAL_SAM2_SETUP = True to install official SAM2 and download checkpoints in this runtime.")
    print("Then use Model Connections -> SAM2 local and diagnose the saved checkpoint/config paths.")


## Optional local SAM3 package setup

Use local SAM3 for concept prompts such as `red ball` or `person in white`. This cell installs the official SAM3 source package only. It clones `https://github.com/facebookresearch/sam3.git` into `/content/sam3` and runs `pip install -e /content/sam3`.

Important: `/content/sam3` is not the model checkpoint path. It is the source/package directory. Run the next cell to resolve or download the `facebook/sam3` checkpoint file and set `SAM3_LOCAL_MODEL`.

SAM3 local setup expects official package/model access and may require a Python/CUDA combination that differs from the default Colab image. If local SAM3 is not ready, use Roboflow SAM3 or Fal SAM3 image from Model Connections.


In [ ]:
RUN_LOCAL_SAM3_SETUP = False

if RUN_LOCAL_SAM3_SETUP:
    if sys.version_info < (3, 12):
        print(f"Warning: Python {sys.version.split()[0]} detected. Local SAM3 currently expects Python 3.12+.")
    if not SAM3_SOURCE_DIR.exists():
        subprocess.run(["git", "clone", "https://github.com/facebookresearch/sam3.git", str(SAM3_SOURCE_DIR)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(SAM3_SOURCE_DIR)], check=True)
    os.environ.setdefault("SAM3_LOCAL_DEVICE", "cuda")
    print(f"Installed SAM3 source/package from {SAM3_SOURCE_DIR}.")
    print("This installs SAM3 code only. It does not download facebook/sam3 sam3.pt.")
    print_sam3_path_help()
else:
    print("Set RUN_LOCAL_SAM3_SETUP = True to clone and install the official SAM3 source package in this runtime.")
    print("This step does not download model checkpoints. Hosted Roboflow SAM3 or Fal SAM3 image can be linked instead.")


## Resolve or download the SAM3 checkpoint path

This cell finds or downloads the real checkpoint file for `SAM3_LOCAL_MODEL`.

- `facebook/sam3` is a Hugging Face repo id. It is useful for `hf_hub_download`, but it is not a local path to paste into MotionJSON.
- The value to paste into Model Connections is the local `sam3.pt` file returned by `hf_hub_download`, usually under `/root/.cache/huggingface/hub/models--facebook--sam3/snapshots/<hash>/sam3.pt`.
- Downloads are opt-in. Leave `RUN_DOWNLOAD_SAM3_CHECKPOINT = False` to avoid surprise large files.

If you already downloaded `sam3.pt`, paste that file path into `MANUAL_SAM3_CHECKPOINT_PATH` and run the cell.


In [ ]:
RUN_DOWNLOAD_SAM3_CHECKPOINT = False
MANUAL_SAM3_CHECKPOINT_PATH = ""  # Example: /root/.cache/huggingface/hub/models--facebook--sam3/snapshots/<hash>/sam3.pt

search_roots = [SAM3_HF_CACHE_DIR, SAM3_SOURCE_DIR]
drive_root = Path("/content/drive/MyDrive")
if drive_root.exists():
    search_roots.append(drive_root)

resolved_model_path: Path | None = None
manual_path = MANUAL_SAM3_CHECKPOINT_PATH.strip()

if manual_path:
    resolved_model_path = set_and_validate_sam3_local_model(manual_path)
    print("Using manually supplied SAM3 checkpoint path.")
else:
    candidates = find_sam3_checkpoint_candidates(search_roots)
    if candidates:
        resolved_model_path = set_and_validate_sam3_local_model(candidates[0])
        print("Found cached SAM3 checkpoint:", resolved_model_path)
    elif RUN_DOWNLOAD_SAM3_CHECKPOINT:
        token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")
        if not token:
            raise RuntimeError("HF_TOKEN or HUGGINGFACE_HUB_TOKEN is required after Hugging Face access is approved for facebook/sam3.")
        try:
            from huggingface_hub import hf_hub_download
        except ImportError:
            subprocess.run([sys.executable, "-m", "pip", "install", "huggingface_hub"], check=True)
            from huggingface_hub import hf_hub_download
        print("Downloading facebook/sam3 sam3.pt with hf_hub_download. This may be a large gated file.")
        print("The Hugging Face token is passed directly and is not printed.")
        downloaded_path = hf_hub_download(repo_id=SAM3_HF_REPO_ID, filename=SAM3_CHECKPOINT_FILENAME, token=token)
        resolved_model_path = set_and_validate_sam3_local_model(downloaded_path)
    else:
        print("No cached sam3.pt checkpoint was found in the usual Colab locations.")
        print("Set RUN_DOWNLOAD_SAM3_CHECKPOINT = True only after confirming Hugging Face access and accepting the large download.")
        print("You can also paste an existing local sam3.pt path into MANUAL_SAM3_CHECKPOINT_PATH.")
        print_sam3_path_help()

if resolved_model_path:
    print("SAM3_LOCAL_MODEL set to:", resolved_model_path)
    print("Checkpoint size:", friendly_size(resolved_model_path))
    print_sam3_path_help(resolved_model_path)


## Validate SAM3 local readiness

Run this before diagnosing local SAM3 in the UI. It checks runtime basics, package importability, CUDA, and whether `SAM3_LOCAL_MODEL` is a real local checkpoint path. It also calls MotionJSON backend diagnostics without making hosted provider calls.

Hosted SAM3 users can skip local readiness failures and configure Roboflow SAM3 or Fal SAM3 image in Model Connections instead.


In [ ]:
from importlib.util import find_spec

def readiness_row(label: str, ok: bool, detail: str) -> None:
    status = "OK" if ok else "CHECK"
    print(f"[{status}] {label}: {detail}")

py_ok = sys.version_info >= (3, 12)
readiness_row("Python", py_ok, f"{sys.version.split()[0]} detected; local SAM3 expects Python 3.12+.")

torch_ok = False
cuda_ok = False
try:
    import torch
    torch_ok = True
    cuda_ok = bool(torch.cuda.is_available())
    detail = f"torch {torch.__version__}; CUDA available: {cuda_ok}"
    if cuda_ok:
        detail += f"; device: {torch.cuda.get_device_name(0)}"
    readiness_row("PyTorch/CUDA", cuda_ok, detail)
except Exception as exc:
    readiness_row("PyTorch/CUDA", False, f"torch import failed: {type(exc).__name__}: {exc}")

sam3_import_ok = find_spec("sam3") is not None
readiness_row("SAM3 package", sam3_import_ok, "Python can import sam3." if sam3_import_ok else "Install the official source package from /content/sam3 first.")

resolved_for_ui: Path | None = None
current_model_value = os.environ.get("SAM3_LOCAL_MODEL", "").strip()
try:
    resolved_for_ui = set_and_validate_sam3_local_model(current_model_value)
    readiness_row("SAM3_LOCAL_MODEL", True, f"{resolved_for_ui} ({friendly_size(resolved_for_ui)})")
except ValueError as exc:
    readiness_row("SAM3_LOCAL_MODEL", False, str(exc))

if resolved_for_ui:
    print_sam3_path_help(resolved_for_ui)
else:
    print_sam3_path_help()

print("Running MotionJSON backend diagnostics without hosted network calls...")
subprocess.run([sys.executable, "-m", "motionjson.cli", "backend", "diagnostics", "--text"], check=False)


In [ ]:
subprocess.run([sys.executable, "examples/make_demo_video.py", "--out", "examples/demo_red_ball.mp4"], check=True)
subprocess.run([sys.executable, "-m", "motionjson.cli", "backend", "diagnostics", "--text"], check=True)
print("Demo video path to register in the UI: examples/demo_red_ball.mp4")
print("In the UI, open Model Connections, select a recommended SAM provider, diagnose setup, then validate the run config.")


In [ ]:
import time
from google.colab import output

port = 8766
ui_proc = subprocess.Popen(
    ["motionjson", "ui", "--no-open", "--host", "127.0.0.1", "--port", str(port)],
    cwd=str(workdir),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
time.sleep(5)
print("MotionJSON UI started with: motionjson ui --no-open --host 127.0.0.1 --port", port)
print("Use Model Connections to link Replicate SAM2 video, Roboflow SAM3, Fal SAM3 image, local SAM2, local SAM3, or a custom endpoint.")
output.serve_kernel_port_as_iframe(port, path="/ui/", height=900)


In [ ]:
from google.colab import output

output.serve_kernel_port_as_window(8766, path="/ui/")


In [ ]:
if ui_proc.poll() is None and ui_proc.stdout is not None:
    print("UI server is still running. Recent logs will appear here only after the process writes more output.")
else:
    print("UI server exited with code", ui_proc.returncode)


In [ ]:
if 'ui_proc' in globals() and ui_proc.poll() is None:
    ui_proc.terminate()
    print("MotionJSON UI stopped.")
